Colab is making it easier than ever to integrate powerful Generative AI capabilities into your projects. We are launching public preview for a simple and intuitive Python library (google.colab.ai) to access state-of-the-art language models directly within Colab environments. All users have free access to most popular LLMs, while paid users have access to a wider selection of models. This means users can spend less time on configuration and set up and more time bringing their ideas to life. With just a few lines of code, you can now perform a variety of tasks:
- Generate text
- Translate languages
- Write creative content
- Categorize text

Happy Coding!


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/googlecolab/colabtools/blob/main/notebooks/Getting_started_with_google_colab_ai.ipynb)

In [20]:
import pdfplumber

def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() or ""
    return text

In [29]:
def extract_with_chain_of_thought(text):
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=2048,
        system="You are an expert at extracting structured data from trade documents for customs declarations. Always reason carefully before extracting.",
        messages=[{
            "role": "user",
            "content": f"""You will extract customs fields from a commercial invoice.

Before extracting, work through these steps:

Step 1 - Document overview: What type of document is this? What trade route does it cover?
Step 2 - Identify key sections: Where are the party details, line items, weights, and values located?
Step 3 - Resolve ambiguities: Are there multiple values or dates? Which is the correct one for customs?
Step 4 - Extract fields: Now extract each field with confidence.

Invoice text:
{text[:4000]}

Work through all 4 steps, then provide the final extraction:
- Shipper name and address
- Consignee name and address
- Invoice number
- Invoice date
- HS Code(s)
- Description of goods
- Declared value and currency
- Country of origin
- Net weight
- Gross weight

⚠️ AI-assisted extraction — verify all fields before submitting to customs authorities."""
        }]
    )
    return response.content[0].text

# Test on your most complex invoice
text = extract_text_from_pdf("/content/synthetic_invoice_003_MEX_GTM.pdf")
result = extract_with_chain_of_thought(text)
print(result)

# Customs Extraction — Step-by-Step Reasoning

---

## Step 1 — Document Overview

This is a **Commercial Invoice (Factura Comercial)** issued by DHL Express for an international shipment. The trade route is **México → Guatemala**, specifically from **Puerto de Veracruz, México** to **Puerto Santo Tomás de Castilla, Izabal, Guatemala**. The Incoterm is **CFR** (Cost and Freight). Transport is by **sea freight** in a 40' HC dry container. The document is flagged as a **synthetic test document** for AI training purposes — not a real transaction.

---

## Step 2 — Identify Key Sections

| Section | Location |
|---|---|
| **Shipper/Exporter** | Upper left block |
| **Consignee** | Upper right block |
| **Notify Party** | Below consignee |
| **Invoice/shipment details** | Header and port block |
| **Line items** | Main table (6 rows) |
| **Weights & volume** | Cargo Summary block |
| **Values** | Bottom summary block |

---

## Step 3 — Resolve Ambiguities

| Ambiguity | Resolution |
|---|-

In [ ]:
!pip install gradio --quiet

import gradio as gr
import anthropic
import pdfplumber
from google.colab import userdata

client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

def extract_from_invoice(pdf_file):
    # Extract text from uploaded PDF
    text = ""
    with pdfplumber.open(pdf_file.name) as pdf:
        for page in pdf.pages:
            text += page.extract_text() or ""

    # Send to Claude
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        system="You are an expert at extracting structured data from trade documents for customs declarations in Latin America.",
        messages=[{
            "role": "user",
            "content": f"""Extract these customs fields from the invoice below.
If a field is not found, write NOT FOUND.

Fields:
- Shipper name and address
- Consignee name and address
- Invoice number
- Invoice date
- HS Code(s)
- Description of goods
- Declared value and currency
- Country of origin
- Net weight
- Gross weight

Invoice text:
{text[:4000]}

⚠️ AI-assisted extraction — verify all fields before submitting to customs authorities."""
        }]
    )
    return response.content[0].text

# Build the UI
demo = gr.Interface(
    fn=extract_from_invoice,
    inputs=gr.File(label="Upload Commercial Invoice (PDF)", file_types=[".pdf"]),
    outputs=gr.Textbox(label="Extracted Customs Fields", lines=25),
    title="Trade Document Intelligence",
    description="AI-powered extraction of customs fields from commercial invoices. Upload any PDF invoice to extract structured data instantly.\n\n⚠️ Always verify results before submitting to customs authorities.",
    examples=None,
    theme="soft"
)

demo.launch(share=True)

In [ ]:
def extract_invoice_fields(text):
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        system="""You are an expert at extracting structured data from
commercial invoices for customs declarations in Latin America.
Be precise. If a field is not found, write NOT FOUND.""",
        messages=[
            {
                "role": "user",
                "content": f"""Extract these customs-relevant fields from the invoice text below.
Return ONLY a structured list, nothing else.

Fields:
- Shipper name and address
- Consignee name and address
- Invoice number
- Invoice date
- HS Code(s)
- Description of goods
- Declared value and currency
- Country of origin
- Net weight
- Gross weight

Invoice text:
{text[:4000]}

⚠️ HUMAN REVIEW REQUIRED: These results are AI-assisted.
Verify all fields before submitting to customs authorities."""
            }
        ]
    )
    return response.content[0].text

# Test it — will return NOT FOUND for most fields since this is a plan doc, not an invoice
# That is expected and correct behavior
result = extract_invoice_fields(text)
print(result)

# In Colab — after uploading the invoice
text = extract_text_from_pdf("/content/synthetic_invoice_001.pdf")
result = extract_invoice_fields(text)
print(result)

text = extract_text_from_pdf("/content/synthetic_invoice_002_COL_SLV.pdf")
result = extract_invoice_fields(text)
print(result)

text = extract_text_from_pdf("/content/synthetic_invoice_003_MEX_GTM.pdf")
result = extract_invoice_fields(text)
print(result)

text = extract_text_from_pdf("/content/synthetic_invoice_004_PER_ESP.pdf")
result = extract_invoice_fields(text)
print(result)

text = extract_text_from_pdf("/content/synthetic_invoice_005_USA_CRI.pdf")
result = extract_invoice_fields(text)
print(result)

text = extract_text_from_pdf("/content/synthetic_invoice_006_CHL_HND.pdf")
result = extract_invoice_fields(text)
print(result)

text = extract_text_from_pdf("/content/synthetic_invoice_007_ESP_SLV.pdf")
result = extract_invoice_fields(text)
print(result)

text = extract_text_from_pdf("/content/synthetic_invoice_008_CHN_GTM.pdf")
result = extract_invoice_fields(text)
print(result)

text = extract_text_from_pdf("/content/synthetic_invoice_009_PAN_NIC.pdf")
result = extract_invoice_fields(text)
print(result)

text = extract_text_from_pdf("/content/synthetic_invoice_010_KOR_HND.pdf")
result = extract_invoice_fields(text)
print(result)


In [ ]:
import sys

if 'google.colab' in sys.modules:
  %pip install anthropic

import anthropic
from pydantic import BaseModel
from typing import Optional
from google.colab import userdata

class Message(BaseModel):
    role: str
    content: str

class Conversation(BaseModel):
    messages: list[Message] = []

    def add_message(self, role: str, content: str):
        msg = Message(role=role, content=content)
        self.messages.append(msg)
        return msg

    def to_api_format(self, ):
        return [{"role": m.role, "content": m.content} for m in self.messages]

client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

convo = Conversation()
system_prompt = "You are an expert in customs and logistics for Central America. Answer clearly and concisely."

print("Chat with your AI (type 'quit' to stop)\n")

while True:
    user_input = input("You: ")
    if user_input.lower() == "quit":
        break

    convo.add_message("user", user_input)

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        system=system_prompt,
        messages=convo.to_api_format()
    )

    reply = response.content[0].text
    convo.add_message("assistant", reply)
    print(f"\nAI: {reply}\n")

print(f"\nConversation had {len(convo.messages)} messages total.")

In [ ]:
!pip install anthropic pdfplumber --quiet

import anthropic
import pdfplumber
from google.colab import userdata

client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

# Test with any PDF — even a random one to start
# Upload a file to Colab first using the file panel on the left

def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() or ""
    return text

# Once you have a PDF uploaded, test it:
text = extract_text_from_pdf("/content/AI_Business_Plan_Final.pdf")
print(text[:2000])

# Send extracted text to Claude and ask it to identify key fields
# This simulates what we'll do with a real invoice

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    system="You are an expert at extracting structured data from trade documents.",
    messages=[
        {
            "role": "user",
            "content": f"""Extract the following fields from this document.
If a field is not found, write 'NOT FOUND'.

Fields to extract:
- Document title
- Author/Organization
- Main goal or purpose
- Key dates mentioned
- Any numerical targets mentioned

Document text:
{text[:3000]}

Return the results as a clean structured list."""
        }
    ]
)

print(response.content[0].text)

In [ ]:
from pydantic import BaseModel
from typing import Optional

class Message(BaseModel):
    role: str
    content: str
    tokens: Optional[int] = None

class Conversation(BaseModel):
    messages: list[Message] = []
    total_tokens: int = 0

    def add_message(self, role: str, content: str):
        self.messages.append(Message(role=role, content=content))

# Test it
msg = Message(role="user", content="Hello AI world!")
print(msg)

convo = Conversation()
convo.add_message("user", "Hello!")
convo.add_message("assistant", "Hi there!")
print(convo)

In [ ]:
import anthropic
from google.colab import userdata

client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

message = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    system="You are a helpful assistant for someone learning AI engineering.",
    messages=[
        {"role": "user", "content": "Hello Claude! I just wrote my first Pydantic model. What should I learn next?"}
    ]
)

print(message.content[0].text)

In [ ]:
!pip install anthropic pydantic python-dotenv httpx

In [ ]:
# @title List available models
from google.colab import ai

ai.list_models()

Choosing a Model
The model names give you a hint about their capabilities and intended use:

Pro: These are the most capable models, ideal for complex reasoning, creative tasks, and detailed analysis.

Flash: These models are optimized for high speed and efficiency, making them great for summarization, chat applications, and tasks requiring rapid responses.

Gemma: These are lightweight, open-weight models suitable for a variety of text generation tasks and are great for experimentation.

In [ ]:
# @title Simple batch generation example
# Only text-to-text input/output is supported
from google.colab import ai

response = ai.generate_text("What is the capital of France?")
print(response)

For longer text generations, you can stream the response. This displays the output token by token as it's generated, rather than waiting for the entire response to complete. This provides a more interactive and responsive experience. To enable this, simply set stream=True.

In [ ]:
# @title Simple streaming example
from google.colab import ai

stream = ai.generate_text("Tell me a short story.", stream=True)
for text in stream:
  print(text, end='')

In [ ]:
#@title Text formatting setup
#code is not necessary for colab.ai, but is useful in fomatting text chunks
import sys

class LineWrapper:
    def __init__(self, max_length=80):
        self.max_length = max_length
        self.current_line_length = 0

    def print(self, text_chunk):
        i = 0
        n = len(text_chunk)
        while i < n:
            start_index = i
            while i < n and text_chunk[i] not in ' \n': # Find end of word
                i += 1
            current_word = text_chunk[start_index:i]

            delimiter = ""
            if i < n: # If not end of chunk, we found a delimiter
                delimiter = text_chunk[i]
                i += 1 # Consume delimiter

            if current_word:
                needs_leading_space = (self.current_line_length > 0)

                # Case 1: Word itself is too long for a line (must be broken)
                if len(current_word) > self.max_length:
                    if needs_leading_space: # Newline if current line has content
                        sys.stdout.write('\n')
                        self.current_line_length = 0
                    for char_val in current_word: # Break the long word
                        if self.current_line_length >= self.max_length:
                            sys.stdout.write('\n')
                            self.current_line_length = 0
                        sys.stdout.write(char_val)
                        self.current_line_length += 1
                # Case 2: Word doesn't fit on current line (print on new line)
                elif self.current_line_length + (1 if needs_leading_space else 0) + len(current_word) > self.max_length:
                    sys.stdout.write('\n')
                    sys.stdout.write(current_word)
                    self.current_line_length = len(current_word)
                # Case 3: Word fits on current line
                else:
                    if needs_leading_space:
                        # Define punctuation that should not have a leading space
                        # when they form an entire "word" (token) following another word.
                        no_leading_space_punctuation = {
                            ",", ".", ";", ":", "!", "?",        # Standard sentence punctuation
                            ")", "]", "}",                     # Closing brackets
                            "'s", "'S", "'re", "'RE", "'ve", "'VE", # Common contractions
                            "'m", "'M", "'ll", "'LL", "'d", "'D",
                            "n't", "N'T",
                            "...", "…"                          # Ellipses
                        }
                        if current_word not in no_leading_space_punctuation:
                            sys.stdout.write(' ')
                            self.current_line_length += 1
                    sys.stdout.write(current_word)
                    self.current_line_length += len(current_word)

            if delimiter == '\n':
                sys.stdout.write('\n')
                self.current_line_length = 0
            elif delimiter == ' ':
                # If line is full and a space delimiter arrives, it implies a wrap.
                if self.current_line_length >= self.max_length:
                    sys.stdout.write('\n')
                    self.current_line_length = 0

        sys.stdout.flush()


In [ ]:
# @title Formatted streaming example
from google.colab import ai

wrapper = LineWrapper()
for chunk in ai.generate_text('Give me a long winded description about the evolution of the Roman Empire.', model_name='google/gemini-2.0-flash', stream=True):
  wrapper.print(chunk)